# Deterministic Baselines: ARIMA, Random Forest, XGBoost

This notebook evaluates deterministic point-forecast models on our 6-asset portfolio, systematically demonstrating their limitations — especially the failure of prediction intervals during crisis periods.

**Setup:** Train on data before 2022-01-01 (4,309 days). Test on 2022-01-01 to 2026-03-16 (1,053 days, of which 251 are crisis-regime days).

All figures and tables generated by `src/python/models/baseline_runner.py`.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from pathlib import Path

FIG = Path('../outputs/figures/baseline')
TBL = Path('../outputs/tables')

---
## Part 1: Stationarity and ARIMA Order Selection

The ADF test confirms that raw prices are non-stationary (p > 0.05 for most) while log-returns are strongly stationary (p = 0.0 for all). KPSS confirms stationarity of returns.

Auto-ARIMA selects low-order models — mostly ARIMA(0,0,1) or ARIMA(1,0,0). GLD selects ARIMA(0,0,0): a white-noise model, meaning no linear predictability at all.

In [ ]:
display(pd.read_csv(TBL / 'stationarity_tests.csv').style.set_caption('Stationarity Tests'))
print()
display(pd.read_csv(TBL / 'arima_orders.csv').style.set_caption('ARIMA Order Selection (auto_arima)'))

---
## Part 2: ARIMA Residual Diagnostics

Even after ARIMA's best-fit removes any linear structure, the residuals **still violate the normality assumption**:

- **SPY residuals**: excess kurtosis = 7.7, Jarque-Bera p = 0. Normality overwhelmingly rejected.
- **Squared residuals show significant autocorrelation** (Ljung-Box p ≈ 0 for SPY), confirming that volatility clustering persists — ARIMA is structurally blind to it.

The QQ-plots show the characteristic S-curve: fat tails remain. The ACF of squared residuals decays slowly — the same volatility clustering pattern we saw in the raw data (Day 2).

In [ ]:
display(Image(filename=str(FIG / 'arima_residuals_spy.png'), width=900))

In [ ]:
display(Image(filename=str(FIG / 'arima_residuals_ibm.png'), width=900))

---
## Part 3: ML Models — Feature Importance

Random Forest and XGBoost were trained on 133 features (lagged returns, rolling statistics, VIX, calendar dummies) with walk-forward evaluation (refit every 60 days).

Top features are dominated by **VIX level**, **recent volatility**, and **lagged returns** — confirming that volatility state is the most informative signal, not the direction of past returns.

In [ ]:
display(Image(filename=str(FIG / 'rf_feature_importance.png'), width=800))

In [ ]:
display(Image(filename=str(FIG / 'xgb_feature_importance.png'), width=800))

ML residuals still show fat tails — the fundamental distributional problem is not solved by flexible nonlinear models.

In [ ]:
display(Image(filename=str(FIG / 'ml_residuals_spy.png'), width=900))

---
## Part 4: Point Forecast Evaluation

The most striking result: **no model significantly outperforms the random walk** (predicted return = 0). ARIMA, RF, and XGBoost all achieve nearly identical RMSE, and directional accuracy hovers around 50–55% — barely better than a coin flip.

This confirms the efficient market hypothesis for daily returns: the conditional mean is essentially unpredictable. The value of modelling lies not in predicting direction, but in **characterising the distribution** — especially its tails and time-varying volatility.

In [ ]:
metrics = pd.read_csv(TBL / 'point_forecast_metrics.csv')
display(metrics.style.format({'RMSE': '{:.6f}', 'MAE': '{:.6f}'}).set_caption('Point Forecast Metrics (Test Set)'))

In [ ]:
display(Image(filename=str(FIG / 'rmse_comparison.png'), width=800))
display(Image(filename=str(FIG / 'rmse_all_assets.png'), width=800))

---
## Part 5: Prediction Interval Failure — The Key Result

### Coverage by Regime

ARIMA's 95% prediction intervals are based on the assumption of constant-variance normal residuals. The coverage table reveals the critical failure:

| Asset | Overall | Crisis | Calm |
|-------|---------|--------|------|
| **TLT** | **92.0%** | **84.5%** | 94.4% |
| **SPY** | 96.1% | **88.4%** | 98.5% |
| AAPL | 96.6% | 94.0% | 97.4% |

**TLT during crisis: only 84.5% coverage** (target: 95%). SPY drops to 88.4%. The prediction intervals are **constant width** — they don't widen during high-volatility regimes, so they fail precisely when accurate risk quantification matters most.

Meanwhile, during calm periods, the PIs are too wide (98.5% coverage for SPY) — the model wastes capital by being overly conservative when volatility is low.

In [ ]:
cov = pd.read_csv(TBL / 'coverage_by_regime.csv')
display(cov.style.set_caption('ARIMA 95% PI Coverage by Regime'))

In [ ]:
display(Image(filename=str(FIG / 'prediction_intervals_spy.png'), width=900))
display(Image(filename=str(FIG / 'prediction_intervals_ibm.png'), width=900))

### Exceedance Clustering

If the prediction intervals were well-calibrated, exceedances (actual returns falling outside the PI) would be independent — each day's exceedance probability should not depend on yesterday's.

Instead, we observe **P(exceed | prev exceed) = 14.6%** vs **P(exceed) = 3.9%** — a **3.8x clustering ratio**. Exceedances come in bursts during volatile periods, not randomly. The longest consecutive exceedance streak is 3 days.

This clustering violates the conditional coverage assumption and is a direct consequence of the model's constant-volatility assumption.

In [ ]:
exc = pd.read_csv(TBL / 'exceedance_analysis.csv')
display(exc.style.set_caption('Exceedance Clustering Analysis — SPY ARIMA 95% PI'))
display(Image(filename=str(FIG / 'exceedance_clustering_spy.png'), width=900))

---
## Summary Dashboard

In [ ]:
display(Image(filename=str(FIG / 'baseline_summary.png'), width=1000))

---
## Conclusions and Motivation for Probabilistic Approach

### 1. Point forecasts are nearly futile
ARIMA, Random Forest, and XGBoost achieve RMSE of 0.0112 on SPY — effectively identical to the random walk baseline (0.0112). Directional accuracy is 50–55%, barely above chance. The conditional mean of daily returns is essentially unpredictable.

### 2. Prediction intervals fail in crisis
ARIMA 95% prediction intervals achieve 96.1% overall coverage on SPY but collapse to **88.4% during crisis periods** (rate_hikes_2022). For TLT, crisis coverage drops to **84.5%**. The intervals are constant-width — they cannot adapt to the regime-dependent volatility we documented in Day 2's EDA.

### 3. Residuals violate model assumptions
ARIMA residuals exhibit excess kurtosis of 7.7 (SPY) and 12.4 (IBM), with Jarque-Bera p = 0 — normality is overwhelmingly rejected. Ljung-Box tests on squared residuals yield p ≈ 0, confirming that heteroskedasticity and volatility clustering persist after ARIMA modelling.

### 4. ML models don't solve the fundamental problem
Random Forest and XGBoost provide marginal improvement in point prediction and useful feature insights (VIX and volatility features dominate), but they offer no principled uncertainty quantification. Their pseudo-intervals from tree variance are uncalibrated and theoretically unjustified.

### 5. What we need
These failures point directly to the requirements for our stochastic volatility framework:
- **Time-varying volatility** that widens prediction intervals during crises (Heston model's mean-reverting variance process)
- **Fat-tailed distributions** that assign realistic probability to extreme events
- **Leverage effect** modelling (negative return-volatility correlation)
- **Calibrated prediction intervals** that maintain correct coverage across regimes

The next phase calibrates GBM, Heston, and Rough Heston models to provide exactly these capabilities.